In [ ]:
import os
import requests
import pandas as pd
import math

service_key = os.environ["SAFETYDATA_API_KEY"]

url = "https://www.safetydata.go.kr/V2/api/DSSP-IF-10942"

num_of_rows = 1000

params = {
    "serviceKey": service_key,
    "returnType": "json",
    "pageNo": 1,
    "numOfRows": num_of_rows
}

res = requests.get(url, params=params)
data = res.json()

total_count = data["totalCount"]
total_pages = math.ceil(total_count / num_of_rows)

print("전체 데이터 수:", total_count)
print("전체 페이지 수:", total_pages)

In [ ]:
all_rows = []

for page in range(1, total_pages + 1):
    params = {
        "serviceKey": service_key,
        "returnType": "json",
        "pageNo": page,
        "numOfRows": num_of_rows
    }

    res = requests.get(url, params=params)
    data = res.json()

    rows = data.get("body", [])
    all_rows.extend(rows)

    print(f"{page}/{total_pages} 페이지 완료, 누적 {len(all_rows)}개")

df = pd.DataFrame(all_rows)

print(df.shape)
print(df.head())
print(df.columns)

In [ ]:
gyeongnam = df[df["RN_DTL_ADRES"].str.contains("경상남도|경남", na=False)]

print(gyeongnam.shape)
print(gyeongnam[["RSTR_NM", "RN_DTL_ADRES", "USE_PSBL_NMPR", "LA", "LO"]].head())

In [ ]:
import re

def extract_sigungu(address):
    match = re.search(r'경상남도\s+([가-힣]+[시군])', str(address))
    if match:
        return match.group(1)
    return None

gyeongnam = df[df["RN_DTL_ADRES"].str.contains("경상남도|경남", na=False)].copy()

gyeongnam["시군"] = gyeongnam["RN_DTL_ADRES"].apply(extract_sigungu)

print(gyeongnam[["RN_DTL_ADRES", "시군"]].head())

In [ ]:
shelter_count = gyeongnam.groupby("시군").agg(
    무더위쉼터수=("RSTR_NM", "count"),
    이용가능인원합계=("USE_PSBL_NMPR", "sum"),
    에어컨수합계=("COLR_HOLD_ARCNDTN", "sum")
).reset_index()

print(shelter_count)

In [ ]:
gyeongnam["USE_PSBL_NMPR"] = pd.to_numeric(gyeongnam["USE_PSBL_NMPR"], errors="coerce").fillna(0)
gyeongnam["COLR_HOLD_ARCNDTN"] = pd.to_numeric(gyeongnam["COLR_HOLD_ARCNDTN"], errors="coerce").fillna(0)

In [ ]:
shelter_count.to_csv("경상남도_시군별_무더위쉼터.csv", index=False, encoding="utf-8-sig")
gyeongnam.to_csv("경상남도_무더위쉼터_상세.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

shelter = pd.read_csv("경상남도_시군별_무더위쉼터.csv")

print(shelter.head())
print(shelter.info())
print(shelter.shape)

In [ ]:
print(shelter["시군"].unique())
print("시군 개수:", shelter["시군"].nunique())

In [ ]:
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

shelter_sorted = shelter.sort_values("무더위쉼터수", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(shelter_sorted["시군"], shelter_sorted["무더위쉼터수"])
plt.xticks(rotation=45)
plt.title("경상남도 시군별 무더위쉼터 수")
plt.xlabel("시군")
plt.ylabel("무더위쉼터 수")
plt.tight_layout()
plt.show()

In [ ]:
shelter_capacity = shelter.sort_values("이용가능인원합계", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(shelter_capacity["시군"], shelter_capacity["이용가능인원합계"])
plt.xticks(rotation=45)
plt.title("경상남도 시군별 무더위쉼터 이용가능인원 합계")
plt.xlabel("시군")
plt.ylabel("이용가능인원 합계")
plt.tight_layout()
plt.show()

In [ ]:
shelter["쉼터1개당평균수용인원"] = shelter["이용가능인원합계"] / shelter["무더위쉼터수"]

shelter_avg = shelter.sort_values("쉼터1개당평균수용인원", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(shelter_avg["시군"], shelter_avg["쉼터1개당평균수용인원"])
plt.xticks(rotation=45)
plt.title("경상남도 시군별 쉼터 1개당 평균 수용인원")
plt.xlabel("시군")
plt.ylabel("쉼터 1개당 평균 수용인원")
plt.tight_layout()
plt.show()

In [ ]:
shelter.to_csv("경상남도_시군별_무더위쉼터_집계.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

pop = pd.read_csv("202605_202605_연령별인구현황_월간.csv", encoding="cp949")

# 행정구역명에서 괄호 안 행정코드 제거
pop["행정구역명"] = pop["행정구역"].str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()

# 경상남도 시군 단위만 남기기
# 예: 경상남도 창원시 = 사용
# 예: 경상남도 창원시 의창구 = 제외
pop["지역단계수"] = pop["행정구역명"].str.split().str.len()
pop_sgg = pop[pop["지역단계수"] == 2].copy()

pop_sgg["시군"] = pop_sgg["행정구역명"].str.split().str[1]

# 숫자 컬럼의 쉼표 제거 후 숫자로 변환
for col in pop_sgg.columns:
    if "2026년05월" in col:
        pop_sgg[col] = (
            pop_sgg[col]
            .astype(str)
            .str.replace(",", "", regex=False)
        )
        pop_sgg[col] = pd.to_numeric(pop_sgg[col], errors="coerce")

# 70세 이상 인구 계산
age70_cols = [
    "2026년05월_계_70~79세",
    "2026년05월_계_80~89세",
    "2026년05월_계_90~99세",
    "2026년05월_계_100세 이상"
]

pop_sgg["70세이상인구"] = pop_sgg[age70_cols].sum(axis=1)
pop_sgg["70세이상비율"] = (
    pop_sgg["70세이상인구"] / pop_sgg["2026년05월_계_총인구수"] * 100
).round(2)

elderly = pop_sgg[[
    "시군",
    "2026년05월_계_총인구수",
    "70세이상인구",
    "70세이상비율"
]].rename(columns={
    "2026년05월_계_총인구수": "전체인구"
})

print(elderly)

elderly.to_csv("경상남도_시군별_고령인구_70세이상_202605.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

shelter = pd.read_csv("경상남도_시군별_무더위쉼터_집계.csv")
elderly = pd.read_csv("경상남도_시군별_고령인구_70세이상_202605.csv")

print(shelter.head())
print(elderly.head())

In [ ]:
print("무더위쉼터 시군")
print(sorted(shelter["시군"].unique()))

print("고령인구 시군")
print(sorted(elderly["시군"].unique()))

In [ ]:
merged = pd.merge(elderly, shelter, on="시군", how="left")

print(merged.head())
print(merged.info())

In [ ]:
print(merged.isnull().sum())

In [ ]:
merged["70세이상_1만명당_쉼터수"] = (
    merged["무더위쉼터수"] / merged["70세이상인구"] * 10000
).round(2)

In [ ]:
merged["쉼터수용률"] = (
    merged["이용가능인원합계"] / merged["70세이상인구"] * 100
).round(2)

In [ ]:
result = merged[[
    "시군",
    "전체인구",
    "70세이상인구",
    "70세이상비율",
    "무더위쉼터수",
    "이용가능인원합계",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률"
]]

print(result)

In [ ]:
result.to_csv(
    "경상남도_시군별_고령인구_무더위쉼터_병합.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

result_sorted = result.sort_values("70세이상_1만명당_쉼터수", ascending=True)

plt.figure(figsize=(12, 6))
plt.bar(result_sorted["시군"], result_sorted["70세이상_1만명당_쉼터수"])
plt.xticks(rotation=45)
plt.title("경상남도 시군별 70세 이상 인구 1만 명당 무더위쉼터 수")
plt.xlabel("시군")
plt.ylabel("70세 이상 1만 명당 쉼터 수")
plt.tight_layout()
plt.show()

In [ ]:
# 70세 이상 1만 명당 쉼터 수가 낮은 지역 Top 5
low_shelter = result.sort_values("70세이상_1만명당_쉼터수").head(5)

print(low_shelter[[
    "시군",
    "70세이상인구",
    "무더위쉼터수",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률"
]])

In [ ]:
!pip install selenium webdriver-manager

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time

url = "https://data.kma.go.kr/climate/heatWave/selectHeatWaveChart.do"

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get(url)

print("브라우저에서 조건을 직접 선택하세요.")
print("예: 기간 2024년 ~ 2024년, 지역/지점 선택, 검색 클릭")
input("검색 결과 표가 화면에 나오면 Enter를 누르세요...")

# 네가 찾은 tbody XPath
tbody_xpath = "/html/body/div[1]/div[1]/div/div[2]/div[3]/div[8]/div[2]/div/table/tbody"

tbody = driver.find_element(By.XPATH, tbody_xpath)
rows = tbody.find_elements(By.TAG_NAME, "tr")

data = []

for row in rows:
    cols = row.find_elements(By.TAG_NAME, "td")
    values = [col.text.strip() for col in cols]
    
    if len(values) == 3:
        data.append(values)

df = pd.DataFrame(data, columns=["연도", "날짜", "지점"])

print(df.head())
print(df.shape)

driver.quit()

In [ ]:
heat_count = df.groupby("지점").size().reset_index(name="폭염일수")

print(heat_count)

In [ ]:
station_to_sigun = {
    "창원": "창원시",
    "통영": "통영시",
    "진주": "진주시",
    "거창": "거창군",
    "합천": "합천군",
    "밀양": "밀양시",
    "산청": "산청군",
    "거제": "거제시",
    "남해": "남해군"
}

heat_count["시군"] = heat_count["지점"].map(station_to_sigun)

heat_result = heat_count[["시군", "지점", "폭염일수"]]

print(heat_result)

In [ ]:
# 부산, 울산처럼 시군 매핑이 안 된 행 제거
heat_result = heat_result.dropna(subset=["시군"]).copy()

print(heat_result)

In [ ]:
heat_result.to_csv(
    "경상남도_시군별_폭염일수_크롤링.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
final = pd.merge(
    result,
    heat_result[["시군", "폭염일수"]],
    on="시군",
    how="inner"
)

print(final)

In [ ]:
# 폭염일수: 높을수록 위험
final["폭염점수"] = final["폭염일수"].rank(ascending=True)

# 70세 이상 비율: 높을수록 위험
final["고령점수"] = final["70세이상비율"].rank(ascending=True)

# 70세 이상 1만 명당 쉼터 수: 낮을수록 위험
final["쉼터부족점수"] = final["70세이상_1만명당_쉼터수"].rank(ascending=False)

# 최종 폭염 취약점수
final["폭염취약점수"] = (
    final["폭염점수"] +
    final["고령점수"] +
    final["쉼터부족점수"]
).round(2)

final_sorted = final.sort_values("폭염취약점수", ascending=False)

print(final_sorted[[
    "시군",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "폭염취약점수"
]])

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 폭염취약점수 기준 정렬
plot_data = final_sorted.sort_values("폭염취약점수", ascending=True)

# 그래프 생성
plt.figure(figsize=(12, 6))

bars = plt.bar(
    plot_data["시군"],
    plot_data["폭염취약점수"]
)

# 제목과 축 설정
plt.title("경상남도 시군별 폭염 취약점수", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("시군", fontsize=12)
plt.ylabel("폭염 취약점수", fontsize=12)

# x축 글자 회전
plt.xticks(rotation=45)

# y축 보조선
plt.grid(axis="y", linestyle="--", alpha=0.4)

# 막대 위 숫자 표시
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# 여백 정리
plt.tight_layout()

# 보고서용 이미지 저장
plt.savefig("경상남도_시군별_폭염취약점수.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
display_table = final_sorted[[
    "시군",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "폭염취약점수"
]].copy()

display_table.insert(0, "순위", range(1, len(display_table) + 1))

display_table = display_table.rename(columns={
    "시군": "지역",
    "폭염일수": "폭염일수(일)",
    "70세이상비율": "70세 이상 비율(%)",
    "70세이상_1만명당_쉼터수": "70세 이상 1만 명당 쉼터 수",
    "폭염취약점수": "폭염 취약점수"
})

display_table

In [ ]:
# 보고서용 최종 결과표 만들기
report_table = final_sorted[[
    "시군",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률",
    "폭염취약점수"
]].copy()

# 순위 추가
report_table.insert(0, "순위", range(1, len(report_table) + 1))

# 컬럼명 보기 좋게 변경
report_table = report_table.rename(columns={
    "시군": "지역",
    "폭염일수": "폭염일수(일)",
    "70세이상비율": "70세 이상 비율(%)",
    "70세이상_1만명당_쉼터수": "70세 이상 1만 명당 쉼터 수",
    "쉼터수용률": "쉼터 수용률(%)",
    "폭염취약점수": "폭염 취약점수"
})

# 소수점 정리
report_table["70세 이상 비율(%)"] = report_table["70세 이상 비율(%)"].round(2)
report_table["70세 이상 1만 명당 쉼터 수"] = report_table["70세 이상 1만 명당 쉼터 수"].round(2)
report_table["쉼터 수용률(%)"] = report_table["쉼터 수용률(%)"].round(2)
report_table["폭염 취약점수"] = report_table["폭염 취약점수"].round(2)

report_table

In [ ]:
def make_level(rank):
    if rank <= 3:
        return "높음"
    elif rank <= 6:
        return "보통"
    else:
        return "낮음"

report_table["취약도 등급"] = report_table["순위"].apply(make_level)

report_table

In [ ]:
report_table.to_excel(
    "경상남도_폭염취약지역_최종순위표.xlsx",
    index=False
)

report_table.to_csv(
    "경상남도_폭염취약지역_최종순위표.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
styled_table = report_table.style\
    .background_gradient(
        subset=["폭염 취약점수"],
        cmap="OrRd"
    )\
    .background_gradient(
        subset=["폭염일수(일)"],
        cmap="YlOrRd"
    )\
    .background_gradient(
        subset=["70세 이상 비율(%)"],
        cmap="YlGnBu"
    )\
    .background_gradient(
        subset=["70세 이상 1만 명당 쉼터 수"],
        cmap="Blues_r"
    )\
    .format({
        "70세 이상 비율(%)": "{:.2f}",
        "70세 이상 1만 명당 쉼터 수": "{:.2f}",
        "쉼터 수용률(%)": "{:.2f}",
        "폭염 취약점수": "{:.2f}"
    })\
    .set_caption("경상남도 시군별 폭염 취약지역 최종 순위")\
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#40466e"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center")
            ]
        }
    ])

styled_table

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

plt.figure(figsize=(10, 7))

plt.scatter(
    final["폭염일수"],
    final["70세이상_1만명당_쉼터수"],
    s=120,
    alpha=0.75
)

# 지역명 표시
for i, row in final.iterrows():
    plt.text(
        row["폭염일수"] + 0.3,
        row["70세이상_1만명당_쉼터수"],
        row["시군"],
        fontsize=10
    )

plt.title("폭염일수와 고령인구 대비 무더위쉼터 공급 수준 비교", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("폭염일수(일)", fontsize=12)
plt.ylabel("70세 이상 인구 1만 명당 무더위쉼터 수", fontsize=12)

plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("폭염일수_쉼터공급수준_산점도.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plot_data = report_table.sort_values("폭염 취약점수", ascending=True).copy()

color_map = {
    "높음": "#d73027",
    "보통": "#fc8d59",
    "낮음": "#91bfdb"
}

colors = plot_data["취약도 등급"].map(color_map)

plt.figure(figsize=(12, 6))

bars = plt.bar(
    plot_data["지역"],
    plot_data["폭염 취약점수"],
    color=colors
)

plt.title("경상남도 시군별 폭염 취약도 등급", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("지역", fontsize=12)
plt.ylabel("폭염 취약점수", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.4)

# 막대 위 점수 표시
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# 범례 직접 생성
import matplotlib.patches as mpatches

legend_handles = [
    mpatches.Patch(color="#d73027", label="높음"),
    mpatches.Patch(color="#fc8d59", label="보통"),
    mpatches.Patch(color="#91bfdb", label="낮음")
]

plt.legend(handles=legend_handles, title="취약도 등급")

plt.tight_layout()
plt.savefig("경상남도_시군별_폭염취약도_등급그래프.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import requests
import folium

# 혹시 report_table을 파일에서 다시 불러올 경우
# report_table = pd.read_csv("경상남도_폭염취약지역_최종순위표.csv")

# =========================
# 1. 경남 지도 GeoJSON 불러오기
# =========================

geo_url = "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_municipalities_geo_simple.json"
geo_json = requests.get(geo_url).json()

# =========================
# 2. GeoJSON에서 경남만 추출
# =========================
# code가 38로 시작하는 지역 = 경상남도
gn_features = [
    f for f in geo_json["features"]
    if f["properties"]["code"].startswith("38")
]

# =========================
# 3. 창원시 구 단위 통합 처리
# =========================
# 지도 파일에는 창원시 의창구, 성산구, 마산합포구 등이 따로 있을 수 있음
# 우리는 시군 단위 분석이므로 모두 '창원시'로 통일
for f in gn_features:
    clean_name = f["properties"]["name"]
    
    if "창원시" in clean_name:
        f["properties"]["name"] = "창원시"
    else:
        f["properties"]["name"] = clean_name

# 다시 경남 GeoJSON 형태로 구성
gn_geo_json = {
    "type": "FeatureCollection",
    "features": gn_features
}

In [ ]:
# =========================
# 4. 분석 결과 딕셔너리 만들기
# =========================

grade_dict = dict(zip(report_table["지역"], report_table["취약도 등급"]))
score_dict = dict(zip(report_table["지역"], report_table["폭염 취약점수"]))
rank_dict = dict(zip(report_table["지역"], report_table["순위"]))
heat_dict = dict(zip(report_table["지역"], report_table["폭염일수(일)"]))
elderly_dict = dict(zip(report_table["지역"], report_table["70세 이상 비율(%)"]))
shelter_dict = dict(zip(report_table["지역"], report_table["70세 이상 1만 명당 쉼터 수"]))

# =========================
# 5. GeoJSON 속성에 분석 결과 추가
# =========================

for f in gn_geo_json["features"]:
    region_name = f["properties"]["name"]
    
    f["properties"]["순위"] = rank_dict.get(region_name, "-")
    f["properties"]["폭염취약점수"] = score_dict.get(region_name, "-")
    f["properties"]["취약도등급"] = grade_dict.get(region_name, "분석대상 아님")
    f["properties"]["폭염일수"] = heat_dict.get(region_name, "-")
    f["properties"]["고령비율"] = elderly_dict.get(region_name, "-")
    f["properties"]["고령인구1만명당쉼터수"] = shelter_dict.get(region_name, "-")

In [ ]:
# =========================
# 6. 취약도 등급별 색상 설정
# =========================

color_dict = {
    "높음": "#d73027",       # 빨강
    "보통": "#fc8d59",       # 주황
    "낮음": "#91bfdb",       # 파랑
    "분석대상 아님": "#dddddd" # 회색
}

def style_function(feature):
    grade = feature["properties"]["취약도등급"]
    
    return {
        "fillColor": color_dict.get(grade, "#dddddd"),
        "color": "#333333",
        "weight": 1,
        "fillOpacity": 0.75
    }

def highlight_function(feature):
    return {
        "fillColor": "#ffff99",
        "color": "#000000",
        "weight": 2,
        "fillOpacity": 0.9
    }

In [ ]:
# =========================
# 7. Folium 지도 생성
# =========================

m = folium.Map(
    location=[35.35, 128.25],   # 경남 중심 좌표
    zoom_start=8,
    tiles="CartoDB positron"
)

folium.GeoJson(
    gn_geo_json,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "name",
            "순위",
            "취약도등급",
            "폭염취약점수",
            "폭염일수",
            "고령비율",
            "고령인구1만명당쉼터수"
        ],
        aliases=[
            "지역",
            "순위",
            "취약도 등급",
            "폭염 취약점수",
            "폭염일수",
            "70세 이상 비율(%)",
            "70세 이상 1만 명당 쉼터 수"
        ],
        localize=True,
        sticky=True
    )
).add_to(m)

In [ ]:
# =========================
# 8. 범례 추가
# =========================

legend_html = """
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 180px;
height: 145px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;
box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
<b>폭염 취약도 등급</b><br><br>
<span style="background-color:#d73027;width:15px;height:15px;display:inline-block;"></span>
&nbsp;높음<br>
<span style="background-color:#fc8d59;width:15px;height:15px;display:inline-block;"></span>
&nbsp;보통<br>
<span style="background-color:#91bfdb;width:15px;height:15px;display:inline-block;"></span>
&nbsp;낮음<br>
<span style="background-color:#dddddd;width:15px;height:15px;display:inline-block;"></span>
&nbsp;분석대상 아님
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# =========================
# 9. 저장
# =========================

m.save("경상남도_폭염취약도_등급지도.html")

m

In [ ]:
import pandas as pd

# 경남 18개 시군에 적용할 대표 관측지점 매핑
sigun_to_station = {
    "창원시": "창원",
    "진주시": "진주",
    "통영시": "통영",
    "사천시": "진주",
    "김해시": "부산",
    "밀양시": "밀양",
    "거제시": "거제",
    "양산시": "울산",
    "의령군": "진주",
    "함안군": "창원",
    "창녕군": "밀양",
    "고성군": "통영",
    "남해군": "남해",
    "하동군": "남해",
    "산청군": "산청",
    "함양군": "거창",
    "거창군": "거창",
    "합천군": "합천"
}

# 지점별 폭염일수를 딕셔너리로 변환
station_heat_dict = dict(zip(heat_count["지점"], heat_count["폭염일수"]))

# 18개 시군 폭염일수 데이터 생성
heat_18 = pd.DataFrame({
    "시군": list(sigun_to_station.keys()),
    "적용지점": list(sigun_to_station.values())
})

heat_18["폭염일수"] = heat_18["적용지점"].map(station_heat_dict)

print(heat_18)
print("결측치 확인")
print(heat_18.isnull().sum())

In [ ]:
final = pd.merge(
    result,
    heat_18,
    on="시군",
    how="left"
)

print(final[["시군", "적용지점", "폭염일수"]])
print(final.isnull().sum())

In [ ]:
print(final.shape)
print(final.columns)

print(final[[
    "시군",
    "적용지점",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률"
]])

print(final.isnull().sum())

In [ ]:
# 폭염일수: 높을수록 위험
final["폭염점수"] = final["폭염일수"].rank(ascending=True)

# 70세 이상 비율: 높을수록 위험
final["고령점수"] = final["70세이상비율"].rank(ascending=True)

# 70세 이상 1만 명당 쉼터 수: 낮을수록 위험
final["쉼터부족점수"] = final["70세이상_1만명당_쉼터수"].rank(ascending=False)

# 최종 폭염 취약점수
final["폭염취약점수"] = (
    final["폭염점수"] +
    final["고령점수"] +
    final["쉼터부족점수"]
).round(2)

# 점수 높은 순서대로 정렬
final_sorted = final.sort_values("폭염취약점수", ascending=False).reset_index(drop=True)

print(final_sorted[[
    "시군",
    "적용지점",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률",
    "폭염취약점수"
]])

In [ ]:
# 보고서용 표 생성
report_table = final_sorted[[
    "시군",
    "적용지점",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률",
    "폭염취약점수"
]].copy()

# 순위 추가
report_table.insert(0, "순위", range(1, len(report_table) + 1))

# 컬럼명 변경
report_table = report_table.rename(columns={
    "시군": "지역",
    "적용지점": "폭염일수 적용지점",
    "폭염일수": "폭염일수(일)",
    "70세이상비율": "70세 이상 비율(%)",
    "70세이상_1만명당_쉼터수": "70세 이상 1만 명당 쉼터 수",
    "쉼터수용률": "쉼터 수용률(%)",
    "폭염취약점수": "폭염 취약점수"
})

# 소수점 정리
report_table["70세 이상 비율(%)"] = report_table["70세 이상 비율(%)"].round(2)
report_table["70세 이상 1만 명당 쉼터 수"] = report_table["70세 이상 1만 명당 쉼터 수"].round(2)
report_table["쉼터 수용률(%)"] = report_table["쉼터 수용률(%)"].round(2)
report_table["폭염 취약점수"] = report_table["폭염 취약점수"].round(2)

# 취약도 등급 부여: 18개 시군 기준
def make_level(rank):
    if rank <= 6:
        return "높음"
    elif rank <= 12:
        return "보통"
    else:
        return "낮음"

report_table["취약도 등급"] = report_table["순위"].apply(make_level)

report_table

In [ ]:
report_table.to_csv(
    "경상남도_18개시군_폭염취약지역_최종순위표.csv",
    index=False,
    encoding="utf-8-sig"
)

report_table.to_excel(
    "경상남도_18개시군_폭염취약지역_최종순위표.xlsx",
    index=False
)

In [ ]:
styled_table = report_table.style\
    .background_gradient(
        subset=["폭염 취약점수"],
        cmap="OrRd"
    )\
    .background_gradient(
        subset=["폭염일수(일)"],
        cmap="YlOrRd"
    )\
    .background_gradient(
        subset=["70세 이상 비율(%)"],
        cmap="YlGnBu"
    )\
    .background_gradient(
        subset=["70세 이상 1만 명당 쉼터 수"],
        cmap="Blues_r"
    )\
    .format({
        "70세 이상 비율(%)": "{:.2f}",
        "70세 이상 1만 명당 쉼터 수": "{:.2f}",
        "쉼터 수용률(%)": "{:.2f}",
        "폭염 취약점수": "{:.2f}"
    })\
    .set_caption("경상남도 18개 시군별 폭염 취약지역 최종 순위")\
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("caption-side", "top")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#40466e"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center")
            ]
        }
    ])

styled_table

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

plot_data = report_table.sort_values("폭염 취약점수", ascending=True).copy()

color_map = {
    "높음": "#d73027",
    "보통": "#fc8d59",
    "낮음": "#91bfdb"
}

colors = plot_data["취약도 등급"].map(color_map)

plt.figure(figsize=(14, 7))

bars = plt.bar(
    plot_data["지역"],
    plot_data["폭염 취약점수"],
    color=colors,
    edgecolor="black",
    linewidth=0.7
)

plt.title("경상남도 18개 시군별 폭염 취약도 등급", fontsize=17, fontweight="bold", pad=18)
plt.xlabel("지역", fontsize=12)
plt.ylabel("폭염 취약점수", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.35)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.3,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

legend_handles = [
    mpatches.Patch(color="#d73027", label="높음"),
    mpatches.Patch(color="#fc8d59", label="보통"),
    mpatches.Patch(color="#91bfdb", label="낮음")
]

plt.legend(handles=legend_handles, title="취약도 등급", loc="upper left")

plt.tight_layout()
plt.savefig("경상남도_18개시군_폭염취약도_등급그래프.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import requests
import folium

# report_table이 이미 있으면 이 줄은 생략 가능
# report_table = pd.read_csv("경상남도_18개시군_폭염취약지역_최종순위표.csv")

# =========================
# 1. 지도 GeoJSON 불러오기
# =========================

geo_url = "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2013/json/skorea_municipalities_geo_simple.json"
geo_json = requests.get(geo_url).json()

# =========================
# 2. 경상남도만 추출
# =========================

gn_features = [
    f for f in geo_json["features"]
    if f["properties"]["code"].startswith("38")
]

# =========================
# 3. 창원시 구 단위 통합
# =========================

for f in gn_features:
    clean_name = f["properties"]["name"]
    
    if "창원시" in clean_name:
        f["properties"]["name"] = "창원시"
    else:
        f["properties"]["name"] = clean_name

gn_geo_json = {
    "type": "FeatureCollection",
    "features": gn_features
}

# =========================
# 4. 분석 결과 딕셔너리 생성
# =========================

grade_dict = dict(zip(report_table["지역"], report_table["취약도 등급"]))
score_dict = dict(zip(report_table["지역"], report_table["폭염 취약점수"]))
rank_dict = dict(zip(report_table["지역"], report_table["순위"]))
station_dict = dict(zip(report_table["지역"], report_table["폭염일수 적용지점"]))
heat_dict = dict(zip(report_table["지역"], report_table["폭염일수(일)"]))
elderly_dict = dict(zip(report_table["지역"], report_table["70세 이상 비율(%)"]))
shelter_dict = dict(zip(report_table["지역"], report_table["70세 이상 1만 명당 쉼터 수"]))
capacity_dict = dict(zip(report_table["지역"], report_table["쉼터 수용률(%)"]))

# =========================
# 5. GeoJSON에 분석 결과 추가
# =========================

for f in gn_geo_json["features"]:
    region_name = f["properties"]["name"]
    
    f["properties"]["순위"] = rank_dict.get(region_name, "-")
    f["properties"]["폭염취약점수"] = score_dict.get(region_name, "-")
    f["properties"]["취약도등급"] = grade_dict.get(region_name, "정보없음")
    f["properties"]["적용지점"] = station_dict.get(region_name, "-")
    f["properties"]["폭염일수"] = heat_dict.get(region_name, "-")
    f["properties"]["고령비율"] = elderly_dict.get(region_name, "-")
    f["properties"]["고령인구1만명당쉼터수"] = shelter_dict.get(region_name, "-")
    f["properties"]["쉼터수용률"] = capacity_dict.get(region_name, "-")

In [ ]:
# =========================
# 6. 색상 설정
# =========================

color_dict = {
    "높음": "#d73027",
    "보통": "#fc8d59",
    "낮음": "#91bfdb",
    "정보없음": "#dddddd"
}

def style_function(feature):
    grade = feature["properties"]["취약도등급"]
    
    return {
        "fillColor": color_dict.get(grade, "#dddddd"),
        "color": "#333333",
        "weight": 1,
        "fillOpacity": 0.75
    }

def highlight_function(feature):
    return {
        "fillColor": "#ffff99",
        "color": "#000000",
        "weight": 2,
        "fillOpacity": 0.9
    }

# =========================
# 7. 지도 생성
# =========================

m = folium.Map(
    location=[35.35, 128.25],
    zoom_start=8,
    tiles="CartoDB positron"
)

folium.GeoJson(
    gn_geo_json,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "name",
            "순위",
            "취약도등급",
            "폭염취약점수",
            "적용지점",
            "폭염일수",
            "고령비율",
            "고령인구1만명당쉼터수",
            "쉼터수용률"
        ],
        aliases=[
            "지역",
            "순위",
            "취약도 등급",
            "폭염 취약점수",
            "폭염일수 적용지점",
            "폭염일수",
            "70세 이상 비율(%)",
            "70세 이상 1만 명당 쉼터 수",
            "쉼터 수용률(%)"
        ],
        localize=True,
        sticky=True
    )
).add_to(m)

# =========================
# 8. 범례 추가
# =========================

legend_html = """
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 190px;
height: 145px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;
box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
">
<b>폭염 취약도 등급</b><br><br>
<span style="background-color:#d73027;width:15px;height:15px;display:inline-block;"></span>
&nbsp;높음<br>
<span style="background-color:#fc8d59;width:15px;height:15px;display:inline-block;"></span>
&nbsp;보통<br>
<span style="background-color:#91bfdb;width:15px;height:15px;display:inline-block;"></span>
&nbsp;낮음<br>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# =========================
# 9. 저장
# =========================

m.save("경상남도_18개시군_폭염취약도_등급지도.html")

m

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 보고서용 이미지 저장 폴더 만들기
os.makedirs("보고서_이미지", exist_ok=True)

In [ ]:
def save_table_image(df, title, filename, max_rows=None):
    """
    pandas DataFrame을 보고서용 표 이미지로 저장하는 함수
    """
    table_df = df.copy()
    
    if max_rows is not None:
        table_df = table_df.head(max_rows)
    
    # 숫자 소수점 정리
    for col in table_df.columns:
        if pd.api.types.is_float_dtype(table_df[col]):
            table_df[col] = table_df[col].round(2)
    
    row_count = len(table_df)
    col_count = len(table_df.columns)
    
    fig_width = max(10, col_count * 1.6)
    fig_height = max(3, row_count * 0.45 + 1.2)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    
    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center"
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.4)
    
    # 디자인 적용
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#555555")
        cell.set_linewidth(0.6)
        
        if row == 0:
            cell.set_facecolor("#40466e")
            cell.set_text_props(color="white", weight="bold")
        else:
            if row % 2 == 0:
                cell.set_facecolor("#f2f2f2")
            else:
                cell.set_facecolor("white")
    
    plt.title(title, fontsize=16, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(f"보고서_이미지/{filename}", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
table1 = elderly[[
    "시군",
    "전체인구",
    "70세이상인구",
    "70세이상비율"
]].copy()

table1 = table1.rename(columns={
    "시군": "지역",
    "전체인구": "전체 인구",
    "70세이상인구": "70세 이상 인구",
    "70세이상비율": "70세 이상 비율(%)"
})

save_table_image(
    table1,
    "경상남도 18개 시군별 70세 이상 인구 산출 결과",
    "표1_고령인구_전처리결과.png"
)

In [ ]:
table2 = shelter_count[[
    "시군",
    "무더위쉼터수",
    "이용가능인원합계",
    "에어컨수합계"
]].copy()

table2 = table2.rename(columns={
    "시군": "지역",
    "무더위쉼터수": "무더위쉼터 수",
    "이용가능인원합계": "이용가능인원 합계",
    "에어컨수합계": "에어컨 수 합계"
})

save_table_image(
    table2,
    "경상남도 18개 시군별 무더위쉼터 집계 결과",
    "표2_무더위쉼터_집계결과.png"
)

In [ ]:
table3 = heat_18[[
    "시군",
    "적용지점",
    "폭염일수"
]].copy()

table3 = table3.rename(columns={
    "시군": "지역",
    "적용지점": "폭염일수 적용지점",
    "폭염일수": "폭염일수(일)"
})

save_table_image(
    table3,
    "경상남도 18개 시군별 폭염일수 대표 관측지점 적용 결과",
    "표3_폭염일수_적용지점_매칭표.png"
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")

steps = [
    "데이터 수집\n폭염일수·고령인구·무더위쉼터",
    "데이터 전처리\n시군명 통일·창원시 통합",
    "지표 산출\n고령비율·쉼터수·폭염일수",
    "취약점수 계산\n폭염점수+고령점수+쉼터부족점수",
    "시각화\n표·그래프·Folium 지도"
]

x_positions = [0.02, 0.22, 0.42, 0.62, 0.82]

for i, (x, text) in enumerate(zip(x_positions, steps)):
    box = FancyBboxPatch(
        (x, 0.35),
        0.16,
        0.35,
        boxstyle="round,pad=0.03",
        linewidth=1.5,
        edgecolor="#333333",
        facecolor="#e8f1fa"
    )
    ax.add_patch(box)
    ax.text(
        x + 0.08,
        0.525,
        text,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )
    
    if i < len(steps) - 1:
        ax.annotate(
            "",
            xy=(x + 0.19, 0.525),
            xytext=(x + 0.16, 0.525),
            arrowprops=dict(arrowstyle="->", linewidth=2, color="#333333")
        )

plt.title("폭염 취약지역 분석 절차", fontsize=18, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("보고서_이미지/그림1_폭염취약지역_분석절차.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
table_merge = final[[
    "시군",
    "적용지점",
    "폭염일수",
    "70세이상비율",
    "70세이상_1만명당_쉼터수",
    "쉼터수용률"
]].copy()

table_merge = table_merge.rename(columns={
    "시군": "지역",
    "적용지점": "적용지점",
    "폭염일수": "폭염일수(일)",
    "70세이상비율": "70세 이상 비율(%)",
    "70세이상_1만명당_쉼터수": "70세 이상 1만 명당 쉼터 수",
    "쉼터수용률": "쉼터 수용률(%)"
})

save_table_image(
    table_merge,
    "고령인구·무더위쉼터·폭염일수 병합 결과",
    "그림2_최종데이터_병합결과.png"
)

In [ ]:
table4 = report_table[[
    "순위",
    "지역",
    "폭염일수 적용지점",
    "폭염일수(일)",
    "70세 이상 비율(%)",
    "70세 이상 1만 명당 쉼터 수",
    "쉼터 수용률(%)",
    "폭염 취약점수",
    "취약도 등급"
]].copy()

save_table_image(
    table4,
    "경상남도 18개 시군별 폭염 취약지역 최종 순위표",
    "표4_폭염취약지역_최종순위표.png"
)

In [ ]:
plot_data = report_table.sort_values("폭염 취약점수", ascending=True).copy()

color_map = {
    "높음": "#d73027",
    "보통": "#fc8d59",
    "낮음": "#91bfdb"
}

colors = plot_data["취약도 등급"].map(color_map)

plt.figure(figsize=(16, 7))

bars = plt.bar(
    plot_data["지역"],
    plot_data["폭염 취약점수"],
    color=colors,
    edgecolor="black",
    linewidth=0.7
)

plt.title("경상남도 18개 시군별 폭염 취약도 등급", fontsize=18, fontweight="bold", pad=18)
plt.xlabel("지역", fontsize=13)
plt.ylabel("폭염 취약점수", fontsize=13)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.35)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.3,
        f"{height:.1f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

legend_handles = [
    mpatches.Patch(color="#d73027", label="높음"),
    mpatches.Patch(color="#fc8d59", label="보통"),
    mpatches.Patch(color="#91bfdb", label="낮음")
]

plt.legend(handles=legend_handles, title="취약도 등급", loc="upper left")

plt.tight_layout()
plt.savefig("보고서_이미지/그림3_폭염취약도_등급막대그래프.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import webbrowser
import os

map_path = os.path.abspath("경상남도_18개시군_폭염취약도_등급지도.html")
webbrowser.open(map_path)